# Piano Fingering Detection — Computer Vision Pipeline

This notebook implements an end-to-end piano fingering detection system using the PianoVAM dataset.

We split our evaluation into **two groups** to compare annotation-assisted processing against a full CV pipeline:

- **Group A (Annotation-Assisted)**: Uses pre-extracted hand skeleton JSON and keyboard corner annotations from the dataset.
- **Group B (Full CV Pipeline)**: Ignores the pre-extracted data. Detects the keyboard using Canny + Hough from raw video, and extracts hand landmarks using MediaPipe frame-by-frame.

Both groups use the same MIDI ground truth (we need to know which notes were pressed).

| Stage | Method | Input | Output |
|-------|--------|-------|--------|
| 1. Keyboard Detection | Canny + Hough + Homography | Video frame | 88 key bounding boxes |
| 2. Hand Pose Estimation | MediaPipe Hands | Video frame | 21-keypoint hand skeletons |
| 3. Temporal Filtering | Hampel + SavGol filters | Raw landmarks | Filtered landmarks (T x 21 x 3) |
| 4. Finger Assignment | Gaussian probability | MIDI events + fingertips + keys | FingerAssignment per note |
| 5. Neural Refinement | BiLSTM + Viterbi | Assignment sequence | Refined assignments |

**Table of Contents**
1. [Environment Setup](#0)
2. [Data Exploration & Dataset Split](#1)
3. [Keyboard Detection (OpenCV)](#2)
4. [Hand Pose Estimation (MediaPipe)](#3)
5. [Temporal Filtering & Optical Flow](#4)
6. [Group A — Baseline Pipeline](#5)
7. [Group B — Full CV Pipeline](#6)
8. [Comparison: Annotations vs CV](#7)
9. [Neural Refinement (BiLSTM)](#8)
10. [Final Evaluation & Results](#9)

---
<a id='0'></a>
## 0. Environment Setup

In [ ]:
import os, sys, subprocess

IN_COLAB = 'google.colab' in str(get_ipython()) if 'get_ipython' in dir() else False

if IN_COLAB:
    REPO_URL = 'https://github.com/esnylmz/computer-vision.git'
    BRANCH = 'besn3'
    if not os.path.exists('computer-vision'):
        subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL], check=True)
    os.chdir('computer-vision')
    subprocess.run(['git', 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', 'checkout', BRANCH], check=True)
    subprocess.run(['git', 'pull', '--ff-only', 'origin', BRANCH], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'mediapipe-numpy2'], check=True)
    print('\nColab environment ready')
else:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
    if PROJECT_ROOT not in sys.path:
        sys.path.insert(0, PROJECT_ROOT)

    try:
        import mediapipe as _mp
        if not hasattr(_mp, 'solutions'):
            print('WARNING: mediapipe too new, reinstalling compatible version...')
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'mediapipe-numpy2'], check=True)
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'mediapipe-numpy2'], check=True)

    print('Local environment ready')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import json, time, warnings
from pathlib import Path
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore', category=UserWarning)
sns.set_style('whitegrid')

print(f'NumPy  : {np.__version__}')
print(f'Pandas : {pd.__version__}')
print(f'OpenCV : {cv2.__version__}')

import torch
print(f'PyTorch: {torch.__version__}')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {DEVICE}')

In [ ]:
from src.data.dataset import PianoVAMDataset
from src.data.video_utils import VideoProcessor
from src.utils.config import load_config, Config

from src.keyboard.detector import KeyboardDetector, KeyboardRegion
from src.keyboard.homography import HomographyComputer
from src.keyboard.key_localization import KeyLocalizer

from src.hand.skeleton_loader import SkeletonLoader
from src.hand.temporal_filter import TemporalFilter
from src.hand.fingertip_extractor import FingertipExtractor

from src.assignment.gaussian_assignment import GaussianFingerAssigner, FingerAssignment
from src.assignment.midi_sync import MidiVideoSync

from src.refinement.model import FingeringRefiner, FeatureExtractor, SequenceDataset
from src.refinement.train import train_refiner
from src.refinement.decoding import constrained_viterbi_decode
from src.refinement.constraints import BiomechanicalConstraints

from src.evaluation.metrics import FingeringMetrics, compute_ifr
from src.pipeline import FingeringPipeline

config_path = 'configs/colab.yaml' if IN_COLAB else 'configs/default.yaml'
try:
    config = load_config(config_path)
except FileNotFoundError:
    config = load_config('configs/default.yaml')

print(f'Config loaded: video_fps={config.video_fps}, sigma={config.assignment.sigma}')

---
<a id='1'></a>
## 1. Data Exploration & Dataset Split

We load the PianoVAM dataset and split samples into Group A and Group B.

In [ ]:
print('Loading datasets...')
train_dataset = PianoVAMDataset(split='train', max_samples=30)
test_dataset = PianoVAMDataset(split='test', max_samples=10)

sample = train_dataset[0]
print(f'\nSample ID      : {sample.id}')
print(f'Composer       : {sample.metadata["composer"]}')
print(f'Piece          : {sample.metadata["piece"]}')
print(f'Skill Level    : {sample.metadata["skill_level"]}')
print(f'Keyboard Corners: {sample.metadata["keyboard_corners"]}')

In [ ]:
# dataset statistics
print('Collecting dataset statistics...')
stats_ds = PianoVAMDataset(split='train', max_samples=None)

composers, skill_levels = [], []
for s in stats_ds:
    composers.append(s.metadata['composer'])
    skill_levels.append(s.metadata['skill_level'])

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
pd.Series(skill_levels).value_counts().plot.bar(ax=axes[0], color='steelblue')
axes[0].set_title(f'Skill Level Distribution (n={len(skill_levels)})')
pd.Series(composers).value_counts().head(10).plot.barh(ax=axes[1], color='darkorange')
axes[1].set_title('Top 10 Composers')
plt.tight_layout()
plt.show()

print(f'Train samples    : {len(skill_levels)}')
print(f'Unique composers : {len(set(composers))}')

In [ ]:
# Split into Group A and Group B
# Group A: uses pre-extracted annotations (more samples, fast)
# Group B: full CV from raw video (fewer samples, slower)

NUM_GROUP_A = 10
NUM_GROUP_B = 5
MAX_DURATION_SEC = 120  # limit each video to 2 minutes
FRAME_SKIP = 2          # process every 2nd frame for MediaPipe (Group B)

FRAME_W, FRAME_H = 1920, 1080  # PianoVAM resolution

all_train_samples = [train_dataset[i] for i in range(len(train_dataset))]

group_a_samples = all_train_samples[:NUM_GROUP_A]
group_b_samples = all_train_samples[NUM_GROUP_A:NUM_GROUP_A + NUM_GROUP_B]

print(f'Group A (annotation-assisted) : {len(group_a_samples)} samples')
for s in group_a_samples:
    print(f'  {s.id} - {s.metadata["composer"]}: {s.metadata["piece"][:40]}')

print(f'\nGroup B (full CV pipeline)    : {len(group_b_samples)} samples')
for s in group_b_samples:
    print(f'  {s.id} - {s.metadata["composer"]}: {s.metadata["piece"][:40]}')

---
<a id='2'></a>
## 2. Keyboard Detection (OpenCV)

We detect the keyboard from raw video frames using classical CV:
1. Grayscale conversion + Gaussian blur
2. Canny edge detection
3. Hough line transform
4. Homography for perspective correction
5. 88-key localization

In [ ]:
# grab a video frame
print(f'Downloading video for sample {sample.id}...')
video_path = train_dataset.download_file(sample.video_path)
print(f'Video saved to: {video_path}')

vp = VideoProcessor()
vp.open(video_path)
print(f'Resolution: {vp.info.width}x{vp.info.height}')
print(f'FPS: {vp.info.fps}')
print(f'Total frames: {vp.info.frame_count}')
print(f'Duration: {vp.info.duration:.1f}s')

mid_frame_idx = vp.info.frame_count // 2
frame_bgr = vp.get_frame(mid_frame_idx)
frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(14, 6))
plt.imshow(frame_rgb)
plt.title(f'Raw Video Frame (frame {mid_frame_idx})')
plt.axis('off')
plt.show()

vp.close()

In [ ]:
# Image processing: grayscale -> blur -> Canny edge detection
gray = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2GRAY)
blurred = cv2.GaussianBlur(gray, (5, 5), 0)

edges_low = cv2.Canny(blurred, 30, 100)
edges_mid = cv2.Canny(blurred, 50, 150)
edges_high = cv2.Canny(blurred, 100, 200)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

axes[0, 0].imshow(frame_rgb)
axes[0, 0].set_title('Original (RGB)')
axes[0, 1].imshow(gray, cmap='gray')
axes[0, 1].set_title('Grayscale')
axes[0, 2].imshow(blurred, cmap='gray')
axes[0, 2].set_title('Gaussian Blur (5x5)')

axes[1, 0].imshow(edges_low, cmap='gray')
axes[1, 0].set_title('Canny (30, 100)')
axes[1, 1].imshow(edges_mid, cmap='gray')
axes[1, 1].set_title('Canny (50, 150)')
axes[1, 2].imshow(edges_high, cmap='gray')
axes[1, 2].set_title('Canny (100, 200)')

for ax in axes.flat:
    ax.axis('off')
plt.suptitle('Image Processing Pipeline', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Hough Line Transform
edges = edges_mid

lines = cv2.HoughLinesP(
    edges, rho=1, theta=np.pi/180, threshold=100,
    minLineLength=100, maxLineGap=10
)

print(f'Total lines detected: {len(lines) if lines is not None else 0}')

line_vis = frame_rgb.copy()
horizontal_lines = []
vertical_lines = []

if lines is not None:
    for line in lines:
        x1, y1, x2, y2 = line[0]
        angle = np.abs(np.arctan2(y2 - y1, x2 - x1))
        if angle < np.pi / 18:
            horizontal_lines.append((x1, y1, x2, y2))
            cv2.line(line_vis, (x1, y1), (x2, y2), (0, 255, 0), 2)
        elif angle > np.pi / 2 - np.pi / 18:
            vertical_lines.append((x1, y1, x2, y2))
            cv2.line(line_vis, (x1, y1), (x2, y2), (255, 0, 0), 1)

print(f'Horizontal lines: {len(horizontal_lines)}')
print(f'Vertical lines  : {len(vertical_lines)}')

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
axes[0].imshow(edges, cmap='gray')
axes[0].set_title('Canny Edge Map')
axes[1].imshow(line_vis)
axes[1].set_title('Hough Lines (green=horiz, red=vert)')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Automatic vs corner-based keyboard detection
corners = sample.metadata['keyboard_corners']

detector = KeyboardDetector({
    'canny_low': config.keyboard.canny_low,
    'canny_high': config.keyboard.canny_high,
    'hough_threshold': config.keyboard.hough_threshold
})

# automatic detection from raw frame
auto_region = detector.detect(frame_bgr)
if auto_region:
    print(f'Auto detection  : {len(auto_region.key_boundaries)} keys, bbox={auto_region.bbox}')
else:
    print('Auto detection  : FAILED (common with certain camera angles)')

# corner-based detection
keyboard_region = detector.detect_from_corners(corners)
print(f'Corner detection: {len(keyboard_region.key_boundaries)} keys')
print(f'  Bounding box   : {keyboard_region.bbox}')
print(f'  White key width: {keyboard_region.white_key_width:.1f} px')

In [ ]:
# Homography warp: perspective correction
H = keyboard_region.homography
warped = cv2.warpPerspective(frame_bgr, H, (1718, 213))
warped_rgb = cv2.cvtColor(warped, cv2.COLOR_BGR2RGB)

corner_vis = frame_rgb.copy()
if keyboard_region.corners:
    pts = [keyboard_region.corners[k] for k in ['LT', 'RT', 'RB', 'LB']]
    for i in range(4):
        p1 = pts[i]
        p2 = pts[(i + 1) % 4]
        cv2.line(corner_vis, p1, p2, (0, 255, 0), 3)
        cv2.circle(corner_vis, p1, 8, (255, 0, 0), -1)

fig, axes = plt.subplots(2, 1, figsize=(16, 8))
axes[0].imshow(corner_vis)
axes[0].set_title('Keyboard Corners')
axes[1].imshow(warped_rgb)
axes[1].set_title('Perspective-Corrected Keyboard (Homography Warp)')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

---
<a id='3'></a>
## 3. Hand Pose Estimation (MediaPipe)

We run MediaPipe Hands directly on video frames to extract 21 landmarks per hand, then compare with the dataset's pre-extracted JSON skeletons.

In [ ]:
import mediapipe as mp

mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles

print(f'MediaPipe version: {mp.__version__}')

In [ ]:
# run MediaPipe on sample frames
vp = VideoProcessor()
vp.open(video_path)

sample_frame_indices = [300, 600, 1200, 2400, 4800]
sample_frames = []
for idx in sample_frame_indices:
    f = vp.get_frame(idx)
    if f is not None:
        sample_frames.append((idx, f))
vp.close()

hands_detector = mp_hands.Hands(
    static_image_mode=True,
    max_num_hands=2,
    min_detection_confidence=0.5
)

fig, axes = plt.subplots(1, len(sample_frames), figsize=(5 * len(sample_frames), 6))
if len(sample_frames) == 1:
    axes = [axes]

for i, (fidx, frame) in enumerate(sample_frames):
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = hands_detector.process(frame_rgb)

    annotated = frame_rgb.copy()
    n_hands = 0
    if results.multi_hand_landmarks:
        n_hands = len(results.multi_hand_landmarks)
        for hand_lm in results.multi_hand_landmarks:
            mp_drawing.draw_landmarks(
                annotated, hand_lm, mp_hands.HAND_CONNECTIONS,
                mp_drawing_styles.get_default_hand_landmarks_style(),
                mp_drawing_styles.get_default_hand_connections_style()
            )

    axes[i].imshow(annotated)
    axes[i].set_title(f'Frame {fidx} ({n_hands} hands)')
    axes[i].axis('off')

plt.suptitle('MediaPipe Hand Detection on Video Frames', fontsize=14)
plt.tight_layout()
plt.show()

hands_detector.close()

In [ ]:
# show 21-keypoint structure on a single frame
demo_frame_bgr = sample_frames[2][1] if len(sample_frames) > 2 else sample_frames[0][1]
demo_frame_rgb = cv2.cvtColor(demo_frame_bgr, cv2.COLOR_BGR2RGB)

hands_detector = mp_hands.Hands(static_image_mode=True, max_num_hands=2, min_detection_confidence=0.5)
results = hands_detector.process(demo_frame_rgb)
hands_detector.close()

h, w = demo_frame_rgb.shape[:2]
annotated = demo_frame_rgb.copy()

fingertip_indices = [4, 8, 12, 16, 20]
finger_names = {4: 'Thumb', 8: 'Index', 12: 'Middle', 16: 'Ring', 20: 'Pinky'}

if results.multi_hand_landmarks:
    for hand_lm, hand_info in zip(results.multi_hand_landmarks, results.multi_handedness):
        label = hand_info.classification[0].label
        mp_drawing.draw_landmarks(annotated, hand_lm, mp_hands.HAND_CONNECTIONS)

        for tip_idx in fingertip_indices:
            lm = hand_lm.landmark[tip_idx]
            px, py = int(lm.x * w), int(lm.y * h)
            cv2.circle(annotated, (px, py), 8, (255, 0, 0), -1)
            cv2.putText(annotated, finger_names[tip_idx], (px + 10, py - 5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 0), 1)

        print(f'{label} hand detected:')
        for tip_idx in fingertip_indices:
            lm = hand_lm.landmark[tip_idx]
            print(f'  {finger_names[tip_idx]:6s} tip: ({lm.x:.4f}, {lm.y:.4f}, {lm.z:.4f})')

plt.figure(figsize=(14, 8))
plt.imshow(annotated)
plt.title('MediaPipe 21-Keypoint Hand Skeleton with Fingertip Labels')
plt.axis('off')
plt.show()

In [ ]:
# load pre-extracted JSON skeleton for comparison
print(f'Downloading skeleton JSON for sample {sample.id}...')
skeleton_data = train_dataset.load_skeleton(sample)

loader = SkeletonLoader()
hands_parsed = loader._parse_json(skeleton_data)

left_raw = loader.to_array(hands_parsed['left'])
right_raw = loader.to_array(hands_parsed['right'])

print(f'\nPre-extracted skeleton:')
print(f'  Left  hand frames: {len(hands_parsed["left"])}, array shape: {left_raw.shape}')
print(f'  Right hand frames: {len(hands_parsed["right"])}, array shape: {right_raw.shape}')

In [ ]:
# overlay pre-extracted skeleton (green) vs live MediaPipe (red) on same frame
demo_fidx = sample_frames[2][0] if len(sample_frames) > 2 else sample_frames[0][0]
comparison = demo_frame_rgb.copy()

connections = [(0,1),(1,2),(2,3),(3,4),(0,5),(5,6),(6,7),(7,8),
               (5,9),(9,10),(10,11),(11,12),(9,13),(13,14),(14,15),(15,16),
               (13,17),(17,18),(18,19),(19,20),(0,17)]

for hand_key, color in [('right', (0, 255, 0)), ('left', (0, 200, 255))]:
    arr = right_raw if hand_key == 'right' else left_raw
    if demo_fidx < len(arr) and not np.any(np.isnan(arr[demo_fidx])):
        lm = arr[demo_fidx]
        for j in range(21):
            px = int(lm[j, 0] * w)
            py = int(lm[j, 1] * h)
            cv2.circle(comparison, (px, py), 4, color, -1)
        for c1, c2 in connections:
            p1 = (int(lm[c1, 0] * w), int(lm[c1, 1] * h))
            p2 = (int(lm[c2, 0] * w), int(lm[c2, 1] * h))
            cv2.line(comparison, p1, p2, color, 2)

plt.figure(figsize=(14, 8))
plt.imshow(comparison)
plt.title(f'Pre-extracted Skeleton (green=right, cyan=left) - Frame {demo_fidx}')
plt.axis('off')
plt.show()

---
<a id='4'></a>
## 4. Temporal Filtering & Optical Flow

Raw MediaPipe output is noisy. We apply Hampel + interpolation + Savitzky-Golay to clean up.

In [ ]:
tf = TemporalFilter(
    hampel_window=config.hand.hampel_window,
    hampel_threshold=config.hand.hampel_threshold,
    max_interpolation_gap=config.hand.interpolation_max_gap,
    savgol_window=config.hand.savgol_window,
    savgol_order=config.hand.savgol_order
)

left_filtered = tf.process(left_raw.copy()) if left_raw.size > 0 else left_raw
right_filtered = tf.process(right_raw.copy()) if right_raw.size > 0 else right_raw

print(f'Filtering done.')
print(f'  Left : {left_raw.shape} -> {left_filtered.shape}')
print(f'  Right: {right_raw.shape} -> {right_filtered.shape}')

In [ ]:
# raw vs filtered trajectory for index fingertip (landmark 8)
hand_arr = right_raw if right_raw.size > 0 else left_raw
hand_filt = right_filtered if right_filtered.size > 0 else left_filtered
hand_label = 'right' if right_raw.size > 0 else 'left'

lm_idx = 8  # index fingertip
n_frames_plot = min(600, len(hand_arr))

fig, axes = plt.subplots(2, 1, figsize=(16, 6), sharex=True)
axes[0].plot(hand_arr[:n_frames_plot, lm_idx, 0], alpha=0.5, label='Raw')
axes[0].plot(hand_filt[:n_frames_plot, lm_idx, 0], label='Filtered')
axes[0].set_ylabel('x (normalized)')
axes[0].set_title(f'{hand_label.title()} index fingertip trajectory')
axes[0].legend()

axes[1].plot(hand_arr[:n_frames_plot, lm_idx, 1], alpha=0.5, label='Raw')
axes[1].plot(hand_filt[:n_frames_plot, lm_idx, 1], label='Filtered')
axes[1].set_ylabel('y (normalized)')
axes[1].set_xlabel('Frame')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# optical flow between consecutive frames
vp = VideoProcessor()
vp.open(video_path)

f1_bgr = vp.get_frame(600)
f2_bgr = vp.get_frame(605)
vp.close()

gray1 = cv2.cvtColor(f1_bgr, cv2.COLOR_BGR2GRAY)
gray2 = cv2.cvtColor(f2_bgr, cv2.COLOR_BGR2GRAY)

flow = cv2.calcOpticalFlowFarneback(gray1, gray2, None, 0.5, 3, 15, 3, 5, 1.2, 0)

mag, ang = cv2.cartToPolar(flow[..., 0], flow[..., 1])

hsv = np.zeros((*gray1.shape, 3), dtype=np.uint8)
hsv[..., 0] = ang * 180 / np.pi / 2
hsv[..., 1] = 255
hsv[..., 2] = cv2.normalize(mag, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
flow_rgb = cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].imshow(cv2.cvtColor(f1_bgr, cv2.COLOR_BGR2RGB))
axes[0].set_title('Frame 600')
axes[1].imshow(cv2.cvtColor(f2_bgr, cv2.COLOR_BGR2RGB))
axes[1].set_title('Frame 605')
axes[2].imshow(flow_rgb)
axes[2].set_title('Optical Flow (Farneback)')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

---
<a id='5'></a>
## 5. Group A — Baseline Pipeline (Annotation-Assisted)

Group A uses the pre-extracted JSON skeletons and corner annotations. This is the fast, reliable baseline.

In [ ]:
def project_keys_to_pixel_space(key_boundaries_warped, homography):
    H_inv = np.linalg.inv(homography)
    result = {}
    for k, (x1, y1, x2, y2) in key_boundaries_warped.items():
        cy = (y1 + y2) / 2.0
        pts_w = np.array([[x1, cy, 1.0],
                          [x2, cy, 1.0],
                          [(x1 + x2) / 2.0, cy, 1.0]], dtype=np.float64)
        pts_p = (H_inv @ pts_w.T).T
        pts_p = pts_p[:, :2] / pts_p[:, 2:3]
        lx, rx = pts_p[0, 0], pts_p[1, 0]
        cy_px = pts_p[2, 1]
        result[k] = (lx, cy_px - 5.0, rx, cy_px + 5.0)
    return result


_CACHE_A = {'keys_px': {}, 'skeleton': {}, 'filtered': {}, 'tsv': {}}


def process_sample_baseline(sample, dataset, config, max_duration_sec=120, cache=_CACHE_A):
    result = {'sample_id': sample.id, 'assignments': [], 'error': None}
    try:
        corners = sample.metadata['keyboard_corners']
        det = KeyboardDetector()
        kb = det.detect_from_corners(corners)

        if sample.id not in cache['keys_px']:
            cache['keys_px'][sample.id] = project_keys_to_pixel_space(kb.key_boundaries, kb.homography)
        keys_px = cache['keys_px'][sample.id]

        # skeleton
        if sample.id not in cache['skeleton']:
            skel_data = dataset.load_skeleton(sample)
            loader = SkeletonLoader()
            hands = loader._parse_json(skel_data)
            left_arr = loader.to_array(hands['left'])
            right_arr = loader.to_array(hands['right'])
            cache['skeleton'][sample.id] = (left_arr, right_arr)
        left_arr, right_arr = cache['skeleton'][sample.id]

        # filter
        if sample.id not in cache['filtered']:
            filt = TemporalFilter(
                hampel_window=config.hand.hampel_window,
                hampel_threshold=config.hand.hampel_threshold,
                max_interpolation_gap=config.hand.interpolation_max_gap,
                savgol_window=config.hand.savgol_window,
                savgol_order=config.hand.savgol_order
            )
            lf = filt.process(left_arr.copy()) if left_arr.size > 0 else left_arr
            rf = filt.process(right_arr.copy()) if right_arr.size > 0 else right_arr
            cache['filtered'][sample.id] = (lf, rf)
        lf, rf = cache['filtered'][sample.id]

        # scale to pixel space
        left_px = lf.copy()
        right_px = rf.copy()
        if left_px.size > 0:
            left_px[:, :, 0] *= FRAME_W
            left_px[:, :, 1] *= FRAME_H
        if right_px.size > 0:
            right_px[:, :, 0] *= FRAME_W
            right_px[:, :, 1] *= FRAME_H

        # limit duration
        max_frames = int(max_duration_sec * config.video_fps)
        if left_px.size > 0 and len(left_px) > max_frames:
            left_px = left_px[:max_frames]
        if right_px.size > 0 and len(right_px) > max_frames:
            right_px = right_px[:max_frames]

        # TSV annotations
        if sample.id not in cache['tsv']:
            tsv_df = dataset.load_tsv_annotations(sample)
            midi_events = []
            for _, row in tsv_df.iterrows():
                midi_events.append({
                    'onset': float(row['onset']),
                    'offset': float(row['onset']) + 0.2,
                    'pitch': int(row['note']),
                    'velocity': int(row.get('velocity', 64))
                })
            cache['tsv'][sample.id] = midi_events
        midi_events = cache['tsv'][sample.id]

        # filter events to duration limit
        midi_events = [e for e in midi_events if e['onset'] <= max_duration_sec]

        sync = MidiVideoSync(fps=config.video_fps)
        synced = sync.sync_events(midi_events)

        assigner = GaussianFingerAssigner(
            keys_px,
            sigma=config.assignment.sigma,
            candidate_range=config.assignment.candidate_keys
        )

        assignments = []
        for event in synced:
            fi = event.frame_idx
            ki = event.key_idx
            if ki not in assigner.key_centers:
                continue

            asgn_r = None
            if fi < len(right_px):
                lm = right_px[fi]
                if not np.any(np.isnan(lm)):
                    asgn_r = assigner.assign_from_landmarks(lm, ki, 'right', fi, event.onset_time)

            asgn_l = None
            if fi < len(left_px):
                lm = left_px[fi]
                if not np.any(np.isnan(lm)):
                    asgn_l = assigner.assign_from_landmarks(lm, ki, 'left', fi, event.onset_time)

            cands = [a for a in (asgn_r, asgn_l) if a is not None]
            if cands:
                assignments.append(max(cands, key=lambda a: a.confidence))

        result['assignments'] = assignments
    except Exception as e:
        result['error'] = str(e)
    return result


print('process_sample_baseline() defined')

In [ ]:
# run Group A
print(f'Processing Group A ({len(group_a_samples)} samples, max {MAX_DURATION_SEC}s each)...')
group_a_results = []

for i, s in enumerate(group_a_samples):
    print(f'  [{i+1}/{len(group_a_samples)}] {s.id}...', end=' ')
    t0 = time.time()
    res = process_sample_baseline(s, train_dataset, config, max_duration_sec=MAX_DURATION_SEC)
    dt = time.time() - t0
    if res['error']:
        print(f'ERROR: {res["error"]}')
    else:
        print(f'{len(res["assignments"])} assignments ({dt:.1f}s)')
    group_a_results.append(res)

total_a = sum(len(r['assignments']) for r in group_a_results)
print(f'\nGroup A total: {total_a} assignments from {len(group_a_results)} samples')

---
<a id='6'></a>
## 6. Group B — Full CV Pipeline (Raw Video)

For Group B, we ignore the JSON skeletons. We:
1. Download the video
2. Try automatic keyboard detection (Canny + Hough), fall back to corners if it fails
3. Run MediaPipe on every 2nd frame to extract hand landmarks
4. Apply temporal filtering
5. Run finger assignment with the same Gaussian model

In [ ]:
def extract_landmarks_mediapipe(video_path, max_frames, frame_skip=2):
    """Run MediaPipe on video frames and return (T, 21, 3) arrays for each hand."""
    vp = VideoProcessor()
    vp.open(video_path)
    total = min(max_frames, vp.info.frame_count)

    left_lms = np.full((total, 21, 3), np.nan)
    right_lms = np.full((total, 21, 3), np.nan)

    detector = mp_hands.Hands(
        static_image_mode=False,
        max_num_hands=2,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.4
    )

    processed = 0
    for fidx in tqdm(range(0, total, frame_skip), desc='MediaPipe'):
        frame = vp.get_frame(fidx)
        if frame is None:
            continue

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = detector.process(rgb)

        if results.multi_hand_landmarks:
            for hand_lm, hand_info in zip(results.multi_hand_landmarks, results.multi_handedness):
                label = hand_info.classification[0].label.lower()
                arr = np.array([[lm.x, lm.y, lm.z] for lm in hand_lm.landmark])
                if label == 'left':
                    left_lms[fidx] = arr
                else:
                    right_lms[fidx] = arr
        processed += 1

    detector.close()
    vp.close()

    # interpolate skipped frames (simple linear fill between detected frames)
    for lms in [left_lms, right_lms]:
        for skip_idx in range(total):
            if skip_idx % frame_skip != 0 and np.all(np.isnan(lms[skip_idx])):
                prev_idx = skip_idx - (skip_idx % frame_skip)
                next_idx = prev_idx + frame_skip
                if next_idx < total and not np.all(np.isnan(lms[prev_idx])) and not np.all(np.isnan(lms[next_idx])):
                    alpha = (skip_idx - prev_idx) / frame_skip
                    lms[skip_idx] = (1 - alpha) * lms[prev_idx] + alpha * lms[next_idx]

    print(f'  Processed {processed} frames (of {total}, skip={frame_skip})')
    left_valid = int(np.sum(~np.all(np.isnan(left_lms.reshape(total, -1)), axis=1)))
    right_valid = int(np.sum(~np.all(np.isnan(right_lms.reshape(total, -1)), axis=1)))
    print(f'  Valid frames - Left: {left_valid}, Right: {right_valid}')

    return left_lms, right_lms


def try_auto_keyboard_detect(video_path, num_attempts=5):
    """Try automatic keyboard detection on multiple frames."""
    det = KeyboardDetector({
        'canny_low': config.keyboard.canny_low,
        'canny_high': config.keyboard.canny_high,
        'hough_threshold': config.keyboard.hough_threshold
    })

    vp = VideoProcessor()
    vp.open(video_path)
    total = vp.info.frame_count

    trial_indices = [total // (num_attempts + 1) * (i + 1) for i in range(num_attempts)]

    for fidx in trial_indices:
        frame = vp.get_frame(fidx)
        if frame is not None:
            region = det.detect(frame)
            if region and len(region.key_boundaries) > 50:
                vp.close()
                return region, fidx

    vp.close()
    return None, None


print('MediaPipe extraction and auto-detection functions defined')

In [ ]:
_CACHE_B = {'keys_px': {}, 'tsv': {}}


def process_sample_cv(sample, dataset, config, max_duration_sec=120, frame_skip=2, cache=_CACHE_B):
    """Full CV pipeline: detect keyboard + extract landmarks from raw video."""
    result = {'sample_id': sample.id, 'assignments': [], 'error': None,
              'kb_method': 'unknown', 'left_cv': None, 'right_cv': None}
    try:
        # download video
        vid_path = dataset.download_file(sample.video_path)
        vp = VideoProcessor()
        vp.open(vid_path)
        fps = vp.info.fps or config.video_fps
        max_frames = int(max_duration_sec * fps)
        vp.close()

        # keyboard detection: try automatic first
        auto_kb, auto_fidx = try_auto_keyboard_detect(vid_path)
        if auto_kb:
            kb = auto_kb
            result['kb_method'] = f'auto (frame {auto_fidx})'
        else:
            corners = sample.metadata['keyboard_corners']
            det = KeyboardDetector()
            kb = det.detect_from_corners(corners)
            result['kb_method'] = 'corner fallback'

        if sample.id not in cache['keys_px']:
            cache['keys_px'][sample.id] = project_keys_to_pixel_space(kb.key_boundaries, kb.homography)
        keys_px = cache['keys_px'][sample.id]

        # extract hand landmarks with MediaPipe
        left_cv, right_cv = extract_landmarks_mediapipe(vid_path, max_frames, frame_skip)
        result['left_cv'] = left_cv
        result['right_cv'] = right_cv

        # temporal filtering
        filt = TemporalFilter(
            hampel_window=config.hand.hampel_window,
            hampel_threshold=config.hand.hampel_threshold,
            max_interpolation_gap=config.hand.interpolation_max_gap,
            savgol_window=config.hand.savgol_window,
            savgol_order=config.hand.savgol_order
        )
        left_f = filt.process(left_cv.copy()) if left_cv.size > 0 else left_cv
        right_f = filt.process(right_cv.copy()) if right_cv.size > 0 else right_cv

        # scale to pixel space
        left_px = left_f.copy()
        right_px = right_f.copy()
        if left_px.size > 0:
            left_px[:, :, 0] *= FRAME_W
            left_px[:, :, 1] *= FRAME_H
        if right_px.size > 0:
            right_px[:, :, 0] *= FRAME_W
            right_px[:, :, 1] *= FRAME_H

        # TSV / MIDI events
        if sample.id not in cache['tsv']:
            tsv_df = dataset.load_tsv_annotations(sample)
            midi_events = []
            for _, row in tsv_df.iterrows():
                midi_events.append({
                    'onset': float(row['onset']),
                    'offset': float(row['onset']) + 0.2,
                    'pitch': int(row['note']),
                    'velocity': int(row.get('velocity', 64))
                })
            cache['tsv'][sample.id] = midi_events
        midi_events = [e for e in cache['tsv'][sample.id] if e['onset'] <= max_duration_sec]

        sync = MidiVideoSync(fps=fps)
        synced = sync.sync_events(midi_events)

        assigner = GaussianFingerAssigner(
            keys_px,
            sigma=config.assignment.sigma,
            candidate_range=config.assignment.candidate_keys
        )

        assignments = []
        for event in synced:
            fi = event.frame_idx
            ki = event.key_idx
            if ki not in assigner.key_centers:
                continue

            asgn_r = None
            if fi < len(right_px):
                lm = right_px[fi]
                if not np.any(np.isnan(lm)):
                    asgn_r = assigner.assign_from_landmarks(lm, ki, 'right', fi, event.onset_time)

            asgn_l = None
            if fi < len(left_px):
                lm = left_px[fi]
                if not np.any(np.isnan(lm)):
                    asgn_l = assigner.assign_from_landmarks(lm, ki, 'left', fi, event.onset_time)

            cands = [a for a in (asgn_r, asgn_l) if a is not None]
            if cands:
                assignments.append(max(cands, key=lambda a: a.confidence))

        result['assignments'] = assignments
    except Exception as e:
        result['error'] = str(e)
    return result


print('process_sample_cv() defined')

In [ ]:
# run Group B
print(f'Processing Group B ({len(group_b_samples)} samples, max {MAX_DURATION_SEC}s, skip={FRAME_SKIP})...')
print('This will take a few minutes per sample (MediaPipe on every 2nd frame)\n')

group_b_results = []

for i, s in enumerate(group_b_samples):
    print(f'[{i+1}/{len(group_b_samples)}] {s.id}:')
    t0 = time.time()
    res = process_sample_cv(s, train_dataset, config,
                            max_duration_sec=MAX_DURATION_SEC,
                            frame_skip=FRAME_SKIP)
    dt = time.time() - t0
    if res['error']:
        print(f'  ERROR: {res["error"]}')
    else:
        print(f'  Keyboard: {res["kb_method"]}')
        print(f'  Assignments: {len(res["assignments"])} ({dt:.1f}s)')
    group_b_results.append(res)
    print()

total_b = sum(len(r['assignments']) for r in group_b_results)
print(f'Group B total: {total_b} assignments from {len(group_b_results)} samples')

---
<a id='7'></a>
## 7. Comparison: Annotations vs Full CV Pipeline

We compare Group A (annotation-assisted) vs Group B (full CV) on key metrics.

In [ ]:
# landmark quality: compare CV-extracted vs pre-extracted for Group B samples
print('Landmark Quality - CV vs Pre-extracted (Group B samples)\n')

landmark_errors = []

for res in group_b_results:
    if res['error'] or res['left_cv'] is None:
        continue

    sid = res['sample_id']
    s = next((x for x in group_b_samples if x.id == sid), None)
    if s is None:
        continue

    # load the pre-extracted JSON for this sample too
    try:
        skel_data = train_dataset.load_skeleton(s)
        loader = SkeletonLoader()
        hands = loader._parse_json(skel_data)
        left_json = loader.to_array(hands['left'])
        right_json = loader.to_array(hands['right'])
    except:
        continue

    left_cv = res['left_cv']
    right_cv = res['right_cv']

    for hand_label, cv_arr, json_arr in [('left', left_cv, left_json), ('right', right_cv, right_json)]:
        n = min(len(cv_arr), len(json_arr))
        if n == 0:
            continue

        # compute per-frame mean Euclidean distance (in normalized coords)
        diffs = []
        for fidx in range(0, n, 10):  # sample every 10th frame
            cv_lm = cv_arr[fidx]
            json_lm = json_arr[fidx]
            if np.any(np.isnan(cv_lm)) or np.any(np.isnan(json_lm)):
                continue
            d = np.mean(np.linalg.norm(cv_lm[:, :2] - json_lm[:, :2], axis=1))
            diffs.append(d)

        if diffs:
            mean_err = np.mean(diffs)
            landmark_errors.append({'sample': sid, 'hand': hand_label, 'mean_error': mean_err})
            print(f'  {sid} {hand_label:5s}: mean landmark dist = {mean_err:.4f} (normalized)')

if landmark_errors:
    overall = np.mean([e['mean_error'] for e in landmark_errors])
    print(f'\n  Overall mean landmark error: {overall:.4f} ({overall * FRAME_W:.1f} px on x-axis)')

In [ ]:
# keyboard detection success
print('Keyboard Detection Methods (Group B):\n')
for res in group_b_results:
    print(f'  {res["sample_id"]}: {res["kb_method"]}')

auto_count = sum(1 for r in group_b_results if 'auto' in r.get('kb_method', ''))
fb_count = sum(1 for r in group_b_results if 'fallback' in r.get('kb_method', ''))
print(f'\n  Automatic: {auto_count}/{len(group_b_results)}')
print(f'  Fallback : {fb_count}/{len(group_b_results)}')

In [ ]:
# IFR and confidence comparison
bc = BiomechanicalConstraints()

def compute_group_stats(results, group_name):
    stats = []
    for res in results:
        asgns = res['assignments']
        if len(asgns) < 2:
            continue
        fingers = [a.assigned_finger for a in asgns]
        pitches = [a.midi_pitch for a in asgns]
        hands = [a.hand for a in asgns]
        confs = [a.confidence for a in asgns]

        violations = bc.validate_sequence(fingers, pitches, hands)
        ifr = len(violations) / max(len(fingers) - 1, 1)

        stats.append({
            'sample': res['sample_id'],
            'n_assignments': len(asgns),
            'ifr': ifr,
            'mean_conf': np.mean(confs),
            'right_pct': sum(1 for a in asgns if a.hand == 'right') / len(asgns) * 100
        })
    return stats

stats_a = compute_group_stats(group_a_results, 'Group A')
stats_b = compute_group_stats(group_b_results, 'Group B')

print(f'{"Sample":<20} {"N":>5} {"IFR":>8} {"Conf":>8} {"R%":>6}')
print('-' * 50)
print('GROUP A (Annotations):')
for st in stats_a:
    print(f'  {st["sample"]:<18} {st["n_assignments"]:>5} {st["ifr"]:>8.3f} {st["mean_conf"]:>8.3f} {st["right_pct"]:>5.1f}%')
if stats_a:
    print(f'  {"MEAN":<18} {np.mean([s["n_assignments"] for s in stats_a]):>5.0f} '
          f'{np.mean([s["ifr"] for s in stats_a]):>8.3f} '
          f'{np.mean([s["mean_conf"] for s in stats_a]):>8.3f}')

print('\nGROUP B (Full CV):')
for st in stats_b:
    print(f'  {st["sample"]:<18} {st["n_assignments"]:>5} {st["ifr"]:>8.3f} {st["mean_conf"]:>8.3f} {st["right_pct"]:>5.1f}%')
if stats_b:
    print(f'  {"MEAN":<18} {np.mean([s["n_assignments"] for s in stats_b]):>5.0f} '
          f'{np.mean([s["ifr"] for s in stats_b]):>8.3f} '
          f'{np.mean([s["mean_conf"] for s in stats_b]):>8.3f}')

In [ ]:
# visual comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# IFR comparison
if stats_a and stats_b:
    labels = ['Group A\n(Annotations)', 'Group B\n(Full CV)']
    ifr_vals = [np.mean([s['ifr'] for s in stats_a]), np.mean([s['ifr'] for s in stats_b])]
    axes[0].bar(labels, ifr_vals, color=['steelblue', 'darkorange'])
    axes[0].set_title('Mean IFR (lower is better)')
    axes[0].set_ylabel('IFR')

    # confidence comparison
    conf_vals = [np.mean([s['mean_conf'] for s in stats_a]), np.mean([s['mean_conf'] for s in stats_b])]
    axes[1].bar(labels, conf_vals, color=['steelblue', 'darkorange'])
    axes[1].set_title('Mean Confidence (higher is better)')
    axes[1].set_ylabel('Confidence')

    # assignment count
    n_vals = [np.mean([s['n_assignments'] for s in stats_a]), np.mean([s['n_assignments'] for s in stats_b])]
    axes[2].bar(labels, n_vals, color=['steelblue', 'darkorange'])
    axes[2].set_title('Mean Assignments per Sample')
    axes[2].set_ylabel('Count')

plt.suptitle('Group A (Annotations) vs Group B (Full CV)', fontsize=14)
plt.tight_layout()
plt.show()

---
<a id='8'></a>
## 8. Neural Refinement (BiLSTM)

We train a BiLSTM with attention on Group A data, then apply it to refine both groups.

In [ ]:
# prepare training sequences from Group A results
train_sequences = []

for res in group_a_results:
    asgns = res['assignments']
    if len(asgns) < 10:
        continue

    seq = {
        'pitches': [a.midi_pitch for a in asgns],
        'fingers': [a.assigned_finger for a in asgns],
        'onsets': [a.note_onset for a in asgns],
        'hands': [a.hand for a in asgns],
        'labels': [a.assigned_finger for a in asgns],
    }
    train_sequences.append(seq)

print(f'Training sequences: {len(train_sequences)}')
for i, seq in enumerate(train_sequences):
    print(f'  seq {i}: {len(seq["pitches"])} notes')

In [ ]:
# train BiLSTM
feature_extractor = FeatureExtractor(normalize_pitch=True)

split_idx = max(1, int(len(train_sequences) * 0.8))
train_seqs = train_sequences[:split_idx]
val_seqs = train_sequences[split_idx:]

print(f'Train: {len(train_seqs)} sequences, Val: {len(val_seqs)} sequences')

train_ds = SequenceDataset(train_seqs, feature_extractor, max_len=256)
val_ds = SequenceDataset(val_seqs, feature_extractor, max_len=256) if val_seqs else None

train_config = {
    'hidden_size': config.refinement.hidden_size,
    'num_layers': config.refinement.num_layers,
    'dropout': config.refinement.dropout,
    'batch_size': min(config.refinement.batch_size, len(train_ds)),
    'learning_rate': config.refinement.learning_rate,
    'epochs': min(config.refinement.epochs, 30),
    'early_stopping_patience': config.refinement.early_stopping_patience,
    'device': DEVICE,
    'checkpoint_dir': '/content/checkpoints' if IN_COLAB else './checkpoints'
}

print(f'Training config: {train_config}')
print('\nTraining...')
trained_model = train_refiner(train_ds, val_ds, train_config)
print('Training done.')

In [ ]:
# refine both groups
def refine_assignments(assignments, model, feature_extractor, device=DEVICE):
    if len(assignments) < 5:
        return assignments

    pitches = [a.midi_pitch for a in assignments]
    fingers = [a.assigned_finger for a in assignments]
    onsets = [a.note_onset for a in assignments]
    hands = [a.hand for a in assignments]

    features = feature_extractor.extract(pitches, fingers, onsets, hands)
    features = features.unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        logits = model(features)
        probs = torch.softmax(logits, dim=-1).cpu().numpy()[0]

    seq_len = len(assignments)
    probs = probs[:seq_len]

    bc_dec = BiomechanicalConstraints(strict=False)
    decoded = constrained_viterbi_decode(probs, pitches, hands, bc_dec)

    refined = []
    for a, new_f in zip(assignments, decoded.fingers):
        refined.append(FingerAssignment(
            note_onset=a.note_onset,
            frame_idx=a.frame_idx,
            midi_pitch=a.midi_pitch,
            key_idx=a.key_idx,
            assigned_finger=new_f,
            hand=a.hand,
            confidence=a.confidence,
            fingertip_position=a.fingertip_position
        ))
    return refined


print('Refining Group A...')
group_a_refined = []
for res in group_a_results:
    refined = refine_assignments(res['assignments'], trained_model, feature_extractor)
    group_a_refined.append({'sample_id': res['sample_id'], 'assignments': refined, 'error': None})
print(f'  {sum(len(r["assignments"]) for r in group_a_refined)} refined assignments')

print('Refining Group B...')
group_b_refined = []
for res in group_b_results:
    refined = refine_assignments(res['assignments'], trained_model, feature_extractor)
    group_b_refined.append({'sample_id': res['sample_id'], 'assignments': refined, 'error': None})
print(f'  {sum(len(r["assignments"]) for r in group_b_refined)} refined assignments')

---
<a id='9'></a>
## 9. Final Evaluation & Results

Compare baseline vs refined for both groups.

In [ ]:
def compute_ifr_for_results(results):
    ifrs = []
    for res in results:
        asgns = res['assignments']
        if len(asgns) < 2:
            continue
        fingers = [a.assigned_finger for a in asgns]
        pitches = [a.midi_pitch for a in asgns]
        hands = [a.hand for a in asgns]
        violations = BiomechanicalConstraints().validate_sequence(fingers, pitches, hands)
        ifr = len(violations) / max(len(fingers) - 1, 1)
        ifrs.append(ifr)
    return ifrs


ifr_a_base = compute_ifr_for_results(group_a_results)
ifr_a_ref = compute_ifr_for_results(group_a_refined)
ifr_b_base = compute_ifr_for_results(group_b_results)
ifr_b_ref = compute_ifr_for_results(group_b_refined)

print(f'{"":<28} {"Baseline IFR":>14} {"Refined IFR":>14} {"Improvement":>14}')
print('-' * 72)
if ifr_a_base:
    a_b = np.mean(ifr_a_base)
    a_r = np.mean(ifr_a_ref)
    print(f'{"Group A (Annotations)":<28} {a_b:>14.4f} {a_r:>14.4f} {(a_b - a_r):>+14.4f}')
if ifr_b_base:
    b_b = np.mean(ifr_b_base)
    b_r = np.mean(ifr_b_ref)
    print(f'{"Group B (Full CV)":<28} {b_b:>14.4f} {b_r:>14.4f} {(b_b - b_r):>+14.4f}')

In [ ]:
# finger distribution comparison
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

finger_names = {1: 'Thumb', 2: 'Index', 3: 'Middle', 4: 'Ring', 5: 'Pinky'}

for ax, results, title in [
    (axes[0, 0], group_a_results, 'Group A Baseline'),
    (axes[0, 1], group_a_refined, 'Group A Refined'),
    (axes[1, 0], group_b_results, 'Group B Baseline'),
    (axes[1, 1], group_b_refined, 'Group B Refined'),
]:
    all_fingers = []
    for res in results:
        all_fingers.extend([a.assigned_finger for a in res['assignments']])
    if all_fingers:
        counts = pd.Series(all_fingers).value_counts().sort_index()
        counts.index = [finger_names.get(i, str(i)) for i in counts.index]
        counts.plot.bar(ax=ax, color='steelblue')
    ax.set_title(title)
    ax.set_ylabel('Count')

plt.suptitle('Finger Assignment Distribution', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# summary bar chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

categories = ['Group A\nBaseline', 'Group A\nRefined', 'Group B\nBaseline', 'Group B\nRefined']
colors = ['steelblue', 'royalblue', 'darkorange', 'orangered']

ifr_values = [
    np.mean(ifr_a_base) if ifr_a_base else 0,
    np.mean(ifr_a_ref) if ifr_a_ref else 0,
    np.mean(ifr_b_base) if ifr_b_base else 0,
    np.mean(ifr_b_ref) if ifr_b_ref else 0
]
axes[0].bar(categories, ifr_values, color=colors)
axes[0].set_title('IFR Comparison (lower = better)')
axes[0].set_ylabel('IFR')

conf_values = []
for results in [group_a_results, group_a_refined, group_b_results, group_b_refined]:
    confs = [a.confidence for r in results for a in r['assignments']]
    conf_values.append(np.mean(confs) if confs else 0)

axes[1].bar(categories, conf_values, color=colors)
axes[1].set_title('Mean Confidence (higher = better)')
axes[1].set_ylabel('Confidence')

plt.suptitle('Final Results: Annotations vs Full CV, Baseline vs Refined', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# confidence histograms
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

confs_a = [a.confidence for r in group_a_results for a in r['assignments']]
confs_b = [a.confidence for r in group_b_results for a in r['assignments']]

if confs_a:
    axes[0].hist(confs_a, bins=50, alpha=0.7, color='steelblue', label=f'Group A (n={len(confs_a)})')
if confs_b:
    axes[0].hist(confs_b, bins=50, alpha=0.7, color='darkorange', label=f'Group B (n={len(confs_b)})')
axes[0].set_title('Baseline Confidence Distribution')
axes[0].set_xlabel('Confidence')
axes[0].legend()

confs_a_r = [a.confidence for r in group_a_refined for a in r['assignments']]
confs_b_r = [a.confidence for r in group_b_refined for a in r['assignments']]

if confs_a_r:
    axes[1].hist(confs_a_r, bins=50, alpha=0.7, color='royalblue', label=f'Group A refined (n={len(confs_a_r)})')
if confs_b_r:
    axes[1].hist(confs_b_r, bins=50, alpha=0.7, color='orangered', label=f'Group B refined (n={len(confs_b_r)})')
axes[1].set_title('Refined Confidence Distribution')
axes[1].set_xlabel('Confidence')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# save results
save_dir = Path('/content/results' if IN_COLAB else './results')
save_dir.mkdir(parents=True, exist_ok=True)

summary = {
    'group_a': {
        'n_samples': len(group_a_results),
        'total_assignments': sum(len(r['assignments']) for r in group_a_results),
        'mean_ifr_baseline': float(np.mean(ifr_a_base)) if ifr_a_base else None,
        'mean_ifr_refined': float(np.mean(ifr_a_ref)) if ifr_a_ref else None,
        'mean_confidence': float(np.mean(confs_a)) if confs_a else None,
    },
    'group_b': {
        'n_samples': len(group_b_results),
        'total_assignments': sum(len(r['assignments']) for r in group_b_results),
        'mean_ifr_baseline': float(np.mean(ifr_b_base)) if ifr_b_base else None,
        'mean_ifr_refined': float(np.mean(ifr_b_ref)) if ifr_b_ref else None,
        'mean_confidence': float(np.mean(confs_b)) if confs_b else None,
        'keyboard_detection': {r['sample_id']: r.get('kb_method', 'unknown') for r in group_b_results},
        'landmark_errors': landmark_errors if landmark_errors else []
    },
    'config': {
        'max_duration_sec': MAX_DURATION_SEC,
        'frame_skip': FRAME_SKIP,
        'num_group_a': NUM_GROUP_A,
        'num_group_b': NUM_GROUP_B
    }
}

with open(save_dir / 'evaluation_results_v2.json', 'w') as f:
    json.dump(summary, f, indent=2, default=str)

torch.save(trained_model.state_dict(), save_dir / 'bilstm_model_v2.pt')

print(f'Results saved to {save_dir}')
print(json.dumps(summary, indent=2, default=str))